In [1]:
import numpy as np
import cupy as cp
import pandas as pd
import igpu, pickle

In [2]:
def to_day(di):
    if di >= 2:
        return "day6"
    else:
        return "day" + str(di + 1)

In [3]:
w = []
h = []
for mi in range(6):
    ws = []
    hs = []
    for di in range(3):
        mouse = "mouse" + str(mi + 1)
        day = to_day(di)
        x = igpu.open_csv_nonull(mouse + "/" + mouse + "_" + day + ".csv")
        w_save = None
        h_save = None
        cost_min = np.inf
        for i in range(10):
            # 50 frame -> 25 frame (10s -> 5s)
            model = igpu.solve(x.T, 20, 25, z_th=x.min(), random_seed=i*100+mi*10+di*2, corr_max=0.3, comp_rate=0.1, Wlim=0.01, Hlim=1.0)
            c = model.reconstruction_cost()
            updated = ""
            if c < cost_min:
                cost_min = c
                updated = "updated"
                w_save = cp.asnumpy(model.W)
                h_save = cp.asnumpy(model.H.T)
            print(mouse, day, "| iter = {:>2} | k = {:>2}, cost = {:.3f}".format(i + 1, model.k, c), updated)
        ws.append(w_save)
        hs.append(h_save)
    w.append(ws)
    h.append(hs)

mouse1 day1 | iter =  1 | k =  4, cost = 25868746.158 updated
mouse1 day1 | iter =  2 | k =  5, cost = 25557637.982 updated
mouse1 day1 | iter =  3 | k =  5, cost = 25295194.271 updated
mouse1 day1 | iter =  4 | k =  4, cost = 25840011.941 
mouse1 day1 | iter =  5 | k =  4, cost = 25875424.237 
mouse1 day1 | iter =  6 | k =  5, cost = 25483959.777 
mouse1 day1 | iter =  7 | k =  4, cost = 25868827.847 
mouse1 day1 | iter =  8 | k =  5, cost = 25723923.528 
mouse1 day1 | iter =  9 | k =  7, cost = 24952428.597 updated
mouse1 day1 | iter = 10 | k =  4, cost = 25818453.025 
mouse1 day2 | iter =  1 | k =  3, cost = 18626300.001 updated
mouse1 day2 | iter =  2 | k =  2, cost = 18925159.355 
mouse1 day2 | iter =  3 | k =  2, cost = 19133284.389 
mouse1 day2 | iter =  4 | k =  4, cost = 17841242.508 updated
mouse1 day2 | iter =  5 | k =  4, cost = 18094343.332 
mouse1 day2 | iter =  6 | k =  5, cost = 17569133.972 updated
mouse1 day2 | iter =  7 | k =  5, cost = 17561958.622 updated
mouse1 da

In [4]:
with open("w5.pkl", "wb") as f:
    pickle.dump(w, f)
with open("h5.pkl", "wb") as f:
    pickle.dump(h, f)

In [5]:
x = []
for mi in range(6):
    mouse = "mouse" + str(mi + 1)
    xs = []
    for di in range(3):
        day = to_day(di)
        xs.append(pd.read_csv(mouse + "/" + mouse + "_" + day + ".csv", header=None).values)
    x.append(xs)

In [6]:
w_single = []
h_single = []
for mi in range(6):
    ws = []
    hs = []
    for di in range(3):
        t_h, k = h[mi][di].shape
        n, _, l = w[mi][di].shape
        win = l - 1
        win2 = 2 * l - 1
        wss = np.zeros((n, k, 2))
        hss = np.zeros((t_h, k))
        x_min = x[mi][di][x[mi][di] > 0].min()
        print("Processing mouse {}, {} ".format(mi + 1, to_day(di)), end="")
        for ki in range(k):
            u = cp.asnumpy(igpu.conv(cp.asarray(w[mi][di][:, ki:ki+1, :]), cp.asarray(h[mi][di][:, ki:ki+1]).T, x_min))
            # for calculation of corrcoef
            umu = u - u.mean(axis=1)[:, np.newaxis]
            umu2 = umu * umu
            # the most significant cell ID
            i = u.sum(axis=1).argmax()
            
            # matrix contains corrcoef
            tmp = np.zeros((n, win2))
            for j in range(n):
                if j == i:
                    tmp[i, win] = 1
                    continue
                den2i0 = umu2[i].sum()
                den2j0 = umu2[j].sum()
                if den2j0 == 0:
                    continue
                tmp[j, win] = (umu[i] * umu[j]).sum() / np.sqrt(den2i0 * den2j0)
                den2i = den2i0
                den2j = den2j0
                for p in range(1, l):
                    den2i -= umu2[i, -p]
                    den2j -= umu2[j, p - 1]
                    tmp[j, win - p] = (umu[i, p:] * umu[j, :-p]).sum() / np.sqrt(den2i * den2j)
                den2i = den2i0
                den2j = den2j0
                for p in range(1, l):
                    den2i -= umu2[i, -p]
                    den2j -= umu2[j, p - 1]
                    tmp[j, win + p] = (umu[i, :-p] * umu[j, p:]).sum() / np.sqrt(den2i * den2j)

            # Determine the positions of cell activities based on the significant cell
            index = tmp.argmax(axis=1)
            index_count = np.array([(index == i).sum() for i in range(win2)])
            scand = np.zeros(win)
            c = index_count[:win].sum()
            for s in range(win):
                c += index_count[s + win]
                scand[s] = c
                c -= index_count[s]
            # starting point of the sequential activity
            start = scand.argmax()

            # Determine W and calculate H inversely.
            # Use the average value for the final H.
            w_pos = tmp[:, start:start+l].argmax(axis=1)
            w_act = u.sum(axis=1) / u.sum()
            h_raw = u / w_act[:, np.newaxis]
            h_count = np.zeros(t_h)
            for i, (a, p) in enumerate(zip(w_act, w_pos)):
                hss[win-p:t_h-p, ki] += a * h_raw[i]
                h_count[win-p:t_h-p] += a
            h_count[h_count == 0] = 1
            hss[:, ki] /= h_count
            wss[:, ki, 0] = w_act
            wss[:, ki, 1] = w_pos
            print(".", end="")
        ws.append(wss)
        hs.append(hss)
        print(" Finished")
    w_single.append(ws)
    h_single.append(hs)

Processing mouse 1, day1 ....... Finished
Processing mouse 1, day2 ..... Finished
Processing mouse 1, day6 ..... Finished
Processing mouse 2, day1 ........ Finished
Processing mouse 2, day2 ....... Finished
Processing mouse 2, day6 ........ Finished
Processing mouse 3, day1 ......... Finished
Processing mouse 3, day2 ....... Finished
Processing mouse 3, day6 ........ Finished
Processing mouse 4, day1 ........ Finished
Processing mouse 4, day2 ...... Finished
Processing mouse 4, day6 ........ Finished
Processing mouse 5, day1 ....... Finished
Processing mouse 5, day2 ..... Finished
Processing mouse 5, day6 ..... Finished
Processing mouse 6, day1 ..... Finished
Processing mouse 6, day2 ..... Finished
Processing mouse 6, day6 ....... Finished


In [7]:
with open("w5_single.pkl", "wb") as f:
    pickle.dump(w_single, f)
with open("h5_single.pkl", "wb") as f:
    pickle.dump(h_single, f)

In [8]:
def moving_average(h, win=25):
    move_ave = np.zeros_like(h)
    s = h[:win].sum(axis=0)
    num = win - 1
    t = move_ave.shape[0]
    for i in range(t):
        if i + win - 1 < t:
            num += 1
            s[:] += h[i+win-1, :]
        move_ave[i, :] = s[:] / num
        if i - (win - 1) > 0:
            num -= 1
            s[:] -= h[i-win+1, :]
    return move_ave

In [9]:
h_spike = []
for mi in range(6):
    spikes = []
    for di in range(3):
        move_ave = moving_average(h_single[mi][di])
        tmp = (h_single[mi][di] - move_ave) / move_ave
        tmp[tmp < 0] = 0
        th =  tmp.mean(axis=0) + 2 * tmp.std(axis=0)
        for i, thi in enumerate(th):
            tmp[:, i][tmp[:, i] > thi] = thi
            tmp[:, i] /= thi
        spikes.append(tmp)
    h_spike.append(spikes)

In [10]:
with open("h5_spike.pkl", "wb") as f:
    pickle.dump(h_spike, f)

In [11]:
with open("beh.pkl", "rb") as f:
    beh = pickle.load(f)